In [ ]:
!pip install transformer-lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=97f7b5f9f9f5f344be772cd8124ce87cd1c96c664191db58c26c83868cc49d73
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-medium")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-medium into HookedTransformer


In [ ]:
def make_repeated_tokens(batch=10, seq_len=50, vocab_size=1000):
    half = seq_len // 2
    tokens = torch.randint(0, vocab_size, (batch, half))
    tokens = torch.cat([tokens, tokens], dim=1)
    return tokens

tokens = make_repeated_tokens()

def get_prev_token_pos(tokens):
    batch, seq_len = tokens.shape
    prev_pos = torch.zeros_like(tokens)

    for b in range(batch):
        seen = {}
        for i in range(seq_len):
            tok = tokens[b, i].item()
            if tok in seen:
                prev_pos[b, i] = seen[tok]
            else:
                prev_pos[b, i] = -1
            seen[tok] = i
    return prev_pos

def induction_score(pattern, prev_pos):

    batch, heads, q_len, k_len = pattern.shape

    scores = torch.zeros(heads)

    for h in range(heads):

        total = 0
        count = 0

        for b in range(batch):

            for t in range(q_len):

                p = prev_pos[b, t].item()


                if p == -1:
                    continue


                if p + 1 >= k_len:
                    continue

                total += pattern[b, h, t, p + 1]

                count += 1

        scores[h] = total / (count + 1e-6)

    return scores

def get_all_induction_scores(model, cache, prev_pos):
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads

    scores = torch.zeros(n_layers, n_heads)

    for layer in range(n_layers):
        pattern = cache["pattern", layer]

        scores[layer] = induction_score(pattern, prev_pos)

    return scores

In [ ]:
tokens = make_repeated_tokens(batch=20, seq_len=50)
logits, cache = model.run_with_cache(tokens)

prev_pos = get_prev_token_pos(tokens)

scores = get_all_induction_scores(model, cache, prev_pos)

def print_top_heads(scores, top_k=10):
    flat = []

    for layer in range(scores.shape[0]):
        for head in range(scores.shape[1]):
            flat.append((scores[layer, head].item(), layer, head))

    flat.sort(reverse=True)

    for score, layer, head in flat[:top_k]:
        print(f"Layer {layer}, Head {head}: {score:.4f}")

print_top_heads(scores)

Layer 11, Head 1: 0.8572
Layer 9, Head 9: 0.8492
Layer 12, Head 1: 0.7873
Layer 7, Head 2: 0.7732
Layer 6, Head 1: 0.7685
Layer 13, Head 14: 0.7116
Layer 18, Head 5: 0.6717
Layer 5, Head 8: 0.6504
Layer 10, Head 1: 0.5020
Layer 20, Head 7: 0.4873


In [ ]:
heads_to_ablate = [
    (11, 1),
    (9, 9),
    (12, 1),
    (6, 1),
    (7, 2),
    (5, 8),
    (13, 14),
    (18, 5)
]

def ablate_heads_hook(z, hook):

    layer = hook.layer()

    for (l, h) in heads_to_ablate:
        if l == layer:
            z[:, :, h, :] = 0

    return z

In [ ]:
prompts = [


    "The capital of India is",
    "The largest planet in the solar system is",

    "Once upon a time there was",
    "In a distant galaxy humanity discovered",


    "The reason climate change is difficult to solve is",
    "Artificial intelligence may become dangerous if",


    "Alice went to Paris. Bob went to London. Alice went to",
    "The cat sat on the mat. The dog sat on the",

    "The true meaning of life is",
    "Materialistic Wealth is not sufficient"
]

def generate_text(
    prompt,
    max_new_tokens=100,
    ablate=False
):

    if not ablate:

        output = model.generate(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=1.0
        )

    else:

        all_fwd_hooks = [
            (
                f"blocks.{layer}.attn.hook_z",
                ablate_heads_hook
            )
            for layer in range(model.cfg.n_layers)
        ]

        with model.hooks(fwd_hooks=all_fwd_hooks):

            output = model.generate(
                prompt,
                max_new_tokens=max_new_tokens,
                temperature=1.0
            )

    return output

In [ ]:
def local_repetition_rate(
    tokens,
    window_size=20
):

    repeated = 0
    total = 0

    for i in range(window_size, len(tokens)):

        current = tokens[i]

        window = tokens[i-window_size:i]

        if current in window:
            repeated += 1

        total += 1

    return repeated / max(total, 1)

In [ ]:
base_scores = []
ablated_scores = []

for prompt in prompts:


    base_text = generate_text(
        prompt,
        ablate=False
    )

    base_tokens = model.to_tokens(base_text)[0].tolist()

    base_lrr = local_repetition_rate(base_tokens)

    base_scores.append(base_lrr)


    ablated_text = generate_text(
        prompt,
        ablate=True
    )

    ablated_tokens = model.to_tokens(ablated_text)[0].tolist()

    ablated_lrr = local_repetition_rate(ablated_tokens)

    ablated_scores.append(ablated_lrr)



    print("=" * 70)
    print("PROMPT:")
    print(prompt)

    print("\nBASE LOCAL REPETITION:")
    print(f"{base_lrr:.4f}")

    print("\nABLATED LOCAL REPETITION:")
    print(f"{ablated_lrr:.4f}")

    print("\nDELTA:")
    print(f"{ablated_lrr - base_lrr:.4f}")


import numpy as np

mean_base = np.mean(base_scores)
mean_ablated = np.mean(ablated_scores)

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"Mean Base LRR      : {mean_base:.4f}")
print(f"Mean Ablated LRR   : {mean_ablated:.4f}")
print(f"Mean Delta         : {mean_ablated - mean_base:.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:
The capital of India is

BASE LOCAL REPETITION:
0.0814

ABLATED LOCAL REPETITION:
0.2442

DELTA:
0.1628


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:
The true meaning of life is

BASE LOCAL REPETITION:
0.1724

ABLATED LOCAL REPETITION:
0.2529

DELTA:
0.0805


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:
Materialisic Wealth is not sufficient

BASE LOCAL REPETITION:
0.0455

ABLATED LOCAL REPETITION:
0.2159

DELTA:
0.1705


FINAL RESULTS
Mean Base LRR      : 0.1459
Mean Ablated LRR   : 0.3313
Mean Delta         : 0.1853
